# CGRP migraine project, Part 2: Chemicals, fair test, and models

Rebuilds everything from ChEMBL in one run. **Run all cells in order** (Runtime -> Run all).
Files are saved to Google Drive (`cgrp_project` folder), so after a Colab restart the data
loads from Drive instead of being rebuilt.

**What happens here**
1. Download chemicals + measured binding strengths for the CGRP receptor from ChEMBL
2. Split them fairly (by scaffold family) and hold the 6 known gepant drugs out
3. Baseline model: fingerprints + random forest
4. ChemBERTa embeddings + ridge
5. Repeat the exam 5 times to see if differences are real
6. MoLFormer embeddings + ridge

**Score:** pActivity = 9 - log10(strength in nM). Each +1 is 10x stronger blocking of the receptor
(9 ~ 1 nM, 7 ~ 100 nM, 5 ~ 10 uM).

## Setup

In [ ]:
%pip install -q rdkit

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import requests
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

# Save everything to Google Drive so a Colab restart can't wipe our progress
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/cgrp_project"
except Exception:
    WORK = "cgrp_project"          # not on Colab: use a local folder
os.makedirs(WORK, exist_ok=True)
print("Saving files to:", WORK)

## Step 1: Chemicals from ChEMBL

Two ChEMBL "folders" hold CGRP-receptor experiments: `CHEMBL3798` (CALCRL alone) and
`CHEMBL2107838` (the CALCRL/RAMP1 complex). We keep clean IC50/Ki/Kd values, take the median
per chemical, and set aside large (MW > 1000) peptide-like molecules.

In [ ]:
CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"
CPD_FILE = f"{WORK}/cgrp_compounds.csv"

def download_activities(chembl_id):
    """Download all binding measurements for one ChEMBL target (it sends them in pages)."""
    url = f"{CHEMBL}/activity.json"
    params = {"target_chembl_id": chembl_id,
              "standard_type__in": "IC50,Ki,Kd",
              "pchembl_value__isnull": "false",
              "limit": 1000}
    rows = []
    while url:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        j = r.json()
        rows += j["activities"]
        nxt = j["page_meta"]["next"]
        url = ("https://www.ebi.ac.uk" + nxt) if nxt else None
        params = None
    return rows

if os.path.exists(CPD_FILE):
    cpd = pd.read_csv(CPD_FILE)
    print("Loaded saved chemicals from Drive:", len(cpd))
else:
    raw = []
    for tid in ["CHEMBL3798", "CHEMBL2107838"]:      # CGRP receptor: CALCRL alone, and CALCRL/RAMP1
        got = download_activities(tid)
        print(tid, "->", len(got), "usable measurements")
        raw += got
    df = pd.DataFrame([{
        "compound_id": a["molecule_chembl_id"],
        "name": a.get("molecule_pref_name"),
        "smiles": a["canonical_smiles"],
        "pactivity": float(a["pchembl_value"]),
    } for a in raw if a.get("canonical_smiles")])
    # one row per chemical: median of its repeated measurements
    cpd = df.groupby(["compound_id", "smiles"], as_index=False).agg(
        name=("name", "first"), pactivity=("pactivity", "median"),
        n_measurements=("pactivity", "size"))
    cpd["mol_weight"] = [Descriptors.MolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan
                         for s in cpd.smiles]
    cpd.to_csv(CPD_FILE, index=False)
    print("Saved chemicals to Drive.")

small = cpd[cpd.mol_weight <= 1000].copy()     # drop big peptide-like molecules
print(f"{len(cpd)} unique chemicals; kept {len(small)} small drug-like molecules "
      f"(set aside {len(cpd) - len(small)} large ones)")

## Step 2: A fair split

- The 6 known gepants are held out completely (bonus test).
- Everything else is split by **scaffold family**, so test chemicals have core structures the model never studied.
- Training chemicals with similarity >= 0.6 to any gepant are removed (labelled `removed`), otherwise the drugs' close relatives would give the answers away.

In [ ]:
known = ["rimegepant", "ubrogepant", "atogepant", "olcegepant", "telcagepant", "zavegepant"]
SPLIT_FILE = f"{WORK}/cgrp_split_final.csv"

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp = lambda s: gen.GetFingerprint(Chem.MolFromSmiles(s))

def scaffold(smi):
    m = Chem.MolFromSmiles(smi)
    return MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else ""

if os.path.exists(SPLIT_FILE):
    final2 = pd.read_csv(SPLIT_FILE)
    final2["scaffold"] = final2.scaffold.fillna("")   # a saved CSV turns "no ring scaffold" into a blank
    gepants = final2[final2.split == "gepant"].copy()
    print("Loaded saved split from Drive.")
else:
    data = small.reset_index(drop=True).copy()

    # 1) set the six known drugs aside
    is_gep = data.name.fillna("").str.lower().isin(known)
    gepants = data[is_gep].copy()
    rest = data[~is_gep].copy().reset_index(drop=True)

    # 2) group the rest by scaffold (core ring skeleton)
    rest["scaffold"] = rest.smiles.map(scaffold)
    groups = rest.groupby("scaffold").indices
    print(f"{len(rest)} chemicals share {len(groups)} scaffolds; biggest family: "
          f"{max(len(v) for v in groups.values())}")

    # 3) deal whole scaffold families into train / val / test (80/10/10)
    rng = np.random.RandomState(42)
    keys = list(groups)
    rng.shuffle(keys)
    keys.sort(key=lambda k: -len(groups[k]))
    share = {"train": 0.8, "val": 0.1, "test": 0.1}
    parts = {"train": [], "val": [], "test": []}
    for k in keys:
        name = max(share, key=lambda s: share[s] * len(rest) - len(parts[s]))
        parts[name].extend(groups[k])
    rest["split"] = ""
    for name, idx in parts.items():
        rest.loc[idx, "split"] = name
    gepants["split"] = "gepant"
    gepants["scaffold"] = gepants.smiles.map(scaffold)
    final = pd.concat([rest, gepants], ignore_index=True)

    # 4) remove training chemicals too similar (>= 0.6) to any known drug, so the drug test is fair
    THRESHOLD = 0.6
    final2 = final.copy()
    train_idx = final2.index[final2.split == "train"]
    tr_fps = [fp(s) for s in final2.loc[train_idx, "smiles"]]
    max_sim = np.zeros(len(tr_fps))
    for _, row in gepants.iterrows():
        max_sim = np.maximum(max_sim, DataStructs.BulkTanimotoSimilarity(fp(row.smiles), tr_fps))
    too_close = train_idx[max_sim >= THRESHOLD]
    final2.loc[too_close, "split"] = "removed"
    print(f"Moved {len(too_close)} gepant look-alikes out of training")
    final2.to_csv(SPLIT_FILE, index=False)
    print("Saved split to Drive.")

print("\nGroup sizes and average strength score:")
for s in ["train", "val", "test", "gepant", "removed"]:
    g = final2[final2.split == s]
    if len(g):
        print(f"  {s:<8} {len(g):>4} chemicals   average score {g.pactivity.mean():.2f}")

## Step 3: Baseline model

Fingerprint (2048 yes/no fragment checklist) + random forest. This is the bar a fancier model has to beat.

In [ ]:
def features(smiles_list):
    """Fingerprint: a 2048-box yes/no checklist of small chemical fragments."""
    return np.array([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in smiles_list],
                    dtype=np.uint8)

d = {s: final2[final2.split == s] for s in ["train", "val", "test", "gepant"]}
X = {s: features(d[s].smiles) for s in d}
y = {s: d[s].pactivity.values for s in d}

rf = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1)
rf.fit(X["train"], y["train"])

def evaluate(tag, predict):
    """Typical miss + ranking skill on val/test, then where the known drugs rank among the test chemicals."""
    print(f"\n--- {tag} ---")
    for nm in ["val", "test"]:
        p = predict(nm)
        miss = np.sqrt(np.mean((p - y[nm]) ** 2))
        print(f"{nm:<5} typical miss {miss:.2f} | ranking skill {spearmanr(p, y[nm])[0]:.2f}")
    pool_pred = np.concatenate([predict("test"), predict("gepant")])
    pool_true = np.concatenate([y["test"], y["gepant"]])
    pred_rank = (-pool_pred).argsort().argsort() + 1
    true_rank = (-pool_true).argsort().argsort() + 1
    n_test = len(y["test"])
    print(f"known drugs among {len(pool_pred)} chemicals (model's rank / true rank):")
    for i, nm in enumerate(d["gepant"].name):
        j = n_test + i
        print(f"  {nm:<12} predicted {pool_pred[j]:.1f} (actual {pool_true[j]:.1f})"
              f" | rank {pred_rank[j]:>3} / {true_rank[j]:>3}")

evaluate("fingerprint + random forest (baseline)", lambda s: rf.predict(X[s]))

## Step 4: ChemBERTa embeddings + ridge

A pretrained chemistry transformer (`DeepChem/ChemBERTa-77M-MLM`) turns each SMILES into a list of numbers;
a simple ridge regression predicts the score from them.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

name = "DeepChem/ChemBERTa-77M-MLM"
tok = AutoTokenizer.from_pretrained(name)
enc = AutoModel.from_pretrained(name).eval().to(dev)

def embed(smiles_list, batch=32):
    """Turn each SMILES into one vector by averaging the transformer's output over the molecule."""
    out = []
    with torch.no_grad():
        for i in range(0, len(smiles_list), batch):
            b = tok(smiles_list[i:i + batch], padding=True, truncation=True,
                    max_length=256, return_tensors="pt").to(dev)
            h = enc(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1)
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out)

In [ ]:
E = {s: embed(list(d[s].smiles)) for s in d}
print("Each chemical is now a list of", E["train"].shape[1], "numbers")

scaler = StandardScaler().fit(E["train"])
ridge = RidgeCV(alphas=np.logspace(-2, 3, 12)).fit(scaler.transform(E["train"]), y["train"])
evaluate("ChemBERTa embeddings + ridge", lambda s: ridge.predict(scaler.transform(E[s])))

## Step 5: Repeat the exam 5 times

One 91-chemical test set can mislead. Five different scaffold splits give an average and a spread.
"Medicines' avg rank" = where the 6 hidden drugs land when mixed into that round's test chemicals
(lower is better; ~19 is perfect, ~90 is random).

In [ ]:
# Repeat the exam 5 times with different scaffold splits, so one lucky/unlucky test set can't mislead us
pool = final2[final2.split.isin(["train", "val", "test"])].reset_index(drop=True)
gep = final2[final2.split == "gepant"].reset_index(drop=True)

Xfp_pool, Xfp_gep = features(pool.smiles), features(gep.smiles)
Xem_pool, Xem_gep = embed(list(pool.smiles)), embed(list(gep.smiles))
y_pool, y_gep = pool.pactivity.values, gep.pactivity.values
groups = pool.scaffold.fillna("").values
print(f"{len(pool)} chemicals in the pool, {len(set(groups))} scaffold families\n")

def run_rounds(name, make_model, Xp, Xg):
    out = []
    for seed in range(5):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(Xp, y_pool, groups))
        model = make_model().fit(Xp[tr], y_pool[tr])
        p, pg = model.predict(Xp[te]), model.predict(Xg)
        pool_pred = np.concatenate([p, pg])
        pool_true = np.concatenate([y_pool[te], y_gep])
        n = len(p)
        out.append({"model": name, "round": seed + 1,
                    "typical miss": np.sqrt(np.mean((p - y_pool[te]) ** 2)),
                    "ranking skill": spearmanr(p, y_pool[te])[0],
                    "medicines' avg rank": ((-pool_pred).argsort().argsort() + 1)[n:].mean(),
                    "perfect avg rank": ((-pool_true).argsort().argsort() + 1)[n:].mean(),
                    "random avg rank": (len(pool_pred) + 1) / 2})
    return out

ridge_model = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))
res = pd.DataFrame(
    run_rounds("fingerprint + random forest",
               lambda: RandomForestRegressor(300, random_state=0, n_jobs=-1), Xfp_pool, Xfp_gep)
    + run_rounds("ChemBERTa + ridge", ridge_model, Xem_pool, Xem_gep))

def summarize(table):
    print("Medicines' average rank in each round (lower is better):")
    print(table.pivot(index="round", columns="model", values="medicines' avg rank").round(1).to_string())
    print("\nAverage over the 5 rounds (± how much it varies):")
    for nm, g in table.groupby("model"):
        print(f"\n{nm}")
        for col in ["typical miss", "ranking skill", "medicines' avg rank"]:
            print(f"  {col:<20} {g[col].mean():.2f} ± {g[col].std():.2f}")
    print(f"\nFor reference: a perfect model would give the medicines an average rank of about "
          f"{table['perfect avg rank'].mean():.0f}; random shuffling gives about {table['random avg rank'].mean():.0f}.")

summarize(res)
res.to_csv(f"{WORK}/results_rounds.csv", index=False)

## Step 6: MoLFormer embeddings + ridge

`ibm/MoLFormer-XL-both-10pct` (pretrained on 10% of ZINC + PubChem). Same five rounds, same splits.

In [ ]:
mname = "ibm/MoLFormer-XL-both-10pct"
mtok = AutoTokenizer.from_pretrained(mname, trust_remote_code=True)
menc = AutoModel.from_pretrained(mname, deterministic_eval=True, trust_remote_code=True).eval().to(dev)

def embed_molformer(smiles_list, batch=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(smiles_list), batch):
            b = mtok(smiles_list[i:i + batch], padding=True, truncation=True,
                     max_length=256, return_tensors="pt").to(dev)
            h = menc(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1)
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out)

In [ ]:
Xmf_pool = embed_molformer(list(pool.smiles))
Xmf_gep = embed_molformer(list(gep.smiles))
print("Each chemical is now a list of", Xmf_pool.shape[1], "numbers\n")

mf = pd.DataFrame(run_rounds("MoLFormer + ridge", ridge_model, Xmf_pool, Xmf_gep))
allres = pd.concat([res, mf], ignore_index=True)
summarize(allres)
allres.to_csv(f"{WORK}/results_rounds_all.csv", index=False)

## Notes and known limits

- **Approved-drug screen failed (earlier run):** scoring ~2,300 approved drugs gave nonsense (cetyl alcohol and digoxin at the top, scores above the maximum in the data). Cause: the model only ever saw CGRP-receptor blockers, so it cannot say "not a blocker", and ridge extrapolates wildly on unfamiliar molecules.
- **Next:** add decoys (chemicals probably not blocking this receptor), restrict predictions to the region the model has seen, then re-screen.
- All data is about one receptor, so the protein sequence is identical in every row; the models here use only the chemical.
- Predictions are leads for lab testing, not evidence of activity.